# 協調フィルタリングのサンプルコード

In [1]:
import datetime
import random
from datetime import datetime, timedelta
from functools import reduce, wraps
from itertools import product
from typing import Any, Callable, Dict, List, Literal, Optional, TypeVar, Union, cast

import numpy as np
import pandas as pd
import polars as pl
from gensim.models import Word2Vec
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import LabelEncoder


DataFrameType = Union[pd.DataFrame, pl.DataFrame]
ReturnTypeLiteral = Literal["pandas", "polars"]

F = TypeVar("F", bound=Callable[..., pl.DataFrame])


## データの生成

In [2]:
# 固定乱数シード
random.seed(42)
np.random.seed(42)

# 顧客IDと属性
customer_ids = [f"user_{i}" for i in range(10)]  # 10名
ages = ["20代", "30代", "40代", "50代"]
genders = ["男性", "女性"]
sites = ["A", "B", "C", "D", "E"]

reference_date = datetime.today().date()
# 学習用データを生成
train_data = []
for _ in range(100):
    user = random.choice(customer_ids)
    site = random.choice(sites)
    age = random.choice(ages)
    gender = random.choice(genders)
    # 過去180日以内のランダムな日付
    click_date = reference_date - timedelta(days=random.randint(0, 180))
    train_data.append((user, age, gender, site, click_date))

original_cols = ["user_id", "age_group", "gender", "site", "click_date"]
train_df = pd.DataFrame(train_data, columns=original_cols)
train_df.sort_values("click_date", ascending=False, inplace=True)

# 特徴量生成のために追加
train_df["click"] = 1

# # 予測対象のユーザー（既存4名＋新規2名）
# predict_users = customer_ids[:4] + ["new_user_1", "new_user_2"]
# predict_age_gender = {
#     "user_0": ("30代", "男性"),
#     "user_1": ("40代", "女性"),
#     "user_2": ("20代", "男性"),
#     "user_3": ("50代", "女性"),
#     "new_user_1": ("30代", "男性"),
#     "new_user_2": ("40代", "女性"),
# }

# # 予測用データ（各ユーザー×5サイト）
# predict_data = []
# future_date = datetime.today().date() + timedelta(days=1)
# for user in predict_users:
#     age, gender = predict_age_gender[user]
#     for site in sites:
#         predict_data.append((user, age, gender, site, future_date))

# predict_df = pd.DataFrame(predict_data, columns=["user_id", "age_group", "gender", "site", "prediction_date"])

In [3]:
print("学習用データ")
print(train_df.shape)
print(train_df.head())

学習用データ
(100, 6)
   user_id age_group gender site  click_date  click
84  user_3       20代     女性    C  2025-05-25      1
48  user_6       30代     女性    A  2025-05-25      1
25  user_4       30代     女性    D  2025-05-25      1
67  user_1       40代     男性    D  2025-05-25      1
36  user_9       30代     男性    E  2025-05-24      1


In [4]:
# print("\n予測用データ")
# print(predict_df.shape)
# print(predict_df.head())

## 顧客または顧客属性／サイト単体と顧客または顧客属性×サイトの相互作用

### ユーティリティ関数の定義

In [5]:
def df_io_polars(
        return_type: ReturnTypeLiteral = "polars"
        ) -> Callable[[F], Callable[..., DataFrameType]]:
    """
    Decorator for feature engineering functions that operate using Polars internally,
    while allowing flexible input/output in either pandas or polars DataFrame format.

    Args:
        return_type (str, optional): Desired output format.
            - "pandas": Return pandas.DataFrame (default)
            - "polars": Return polars.DataFrame

    Returns:
        A wrapped function that:
        - Accepts pandas or polars DataFrame as first argument.
        - Converts input to polars internally.
        - Executes the original function (expects polars.DataFrame).
        - Converts output to the specified return_type.
    
    Raises:
        TypeError: If input is neither pandas nor polars DataFrame.
        ValueError: If return_type is invalid.
    """

    def decorator(func: F) -> Callable[..., DataFrameType]:
        @wraps(func)
        def wrapper(df: DataFrameType, *args, **kwargs) -> DataFrameType:
            if not isinstance(df, (pd.DataFrame, pl.DataFrame)):
                raise TypeError(
                    f"[df_io_polars] Expected pandas or \
                      polars DataFrame as first argument, got {type(df)}"
                )

            # Convert to polars if needed
            df_polars = pl.from_pandas(df) if isinstance(df, pd.DataFrame) else df

            # Execute core logic
            result_polars = func(df_polars, *args, **kwargs)

            # Output conversion
            if return_type == "pandas":
                return to_pandas(result_polars)
            elif return_type == "polars":
                return result_polars
            else:
                raise ValueError(
                    f"[df_io_polars] return_type must be 'pandas' or 'polars', got '{return_type}'"
                    )

        return cast(Callable[..., DataFrameType], wrapper)

    return decorator


def to_polars(df: DataFrameType) -> pl.DataFrame:
    """
    Converts a pandas or polars DataFrame to polars.DataFrame.
    """
    if isinstance(df, pd.DataFrame):
        return pl.from_pandas(df)
    elif isinstance(df, pl.DataFrame):
        return df
    else:
        raise TypeError(f"Expected pd.DataFrame or pl.DataFrame, got {type(df)}")


def to_pandas(df: DataFrameType) -> pd.DataFrame:
    """
    Converts a pandas or polars DataFrame to pandas.DataFrame.
    """
    if isinstance(df, pl.DataFrame):
        return df.to_pandas()
    elif isinstance(df, pd.DataFrame):
        return df
    else:
        raise TypeError(f"Expected pd.DataFrame or pl.DataFrame, got {type(df)}")


@df_io_polars(return_type="pandas")
def rename_columns_with_tag(
    df: pl.DataFrame,
    cols: List[str],
    tag: str
) -> pl.DataFrame:
    """
    Rename specified columns in a Polars DataFrame by appending a tag.

    Args:
        df (pl.DataFrame): Input DataFrame.
        cols (List[str]): List of column names to rename.
        tag (str): Suffix tag to append.

    Returns:
        pl.DataFrame: DataFrame with renamed columns.
    """
    rename_map = {col: f"{col}_{tag}" for col in cols if col in df.columns}
    return df.rename(rename_map)


### 特徴量エンジニアリング関数の定義

In [6]:
@df_io_polars(return_type="pandas")
def aggregate_by_category_pivot(
    df: pl.DataFrame,
    index_col: Union[str, List[str]],
    category_col: str,
    agg_func_map: Dict[str, pl.Expr],
    default_value: float = 0.0
) -> pl.DataFrame:
    """
    Wide-format aggregation with multi-index support and safe join (preserves row alignment).

    Returns:
        Polars DataFrame with wide-format stat columns, properly aligned to index_col.
    """
    if isinstance(index_col, str):
        index_col = [index_col]

    result = None

    for stat_name, expr in agg_func_map.items():
        value_col = expr.meta.root_names()[0] if expr.meta.root_names() else "value"

        grouped = df.group_by(index_col + [category_col]).agg(expr.alias("stat"))

        pivoted = grouped.pivot(
            values="stat",
            index=index_col,
            columns=category_col
        )

        rename_dict = {
            col: f"{col}_{value_col}_{stat_name}" if col not in index_col else col
            for col in pivoted.columns
        }
        pivoted = pivoted.rename(rename_dict)

        if result is None:
            result = pivoted
        else:
            # drop conflicting columns before joining
            dup_cols = set(result.columns) & set(pivoted.columns) - set(index_col)
            if dup_cols:
                pivoted = pivoted.drop(list(dup_cols))
            result = result.join(pivoted, on=index_col, how="left")

    return result.fill_null(default_value).sort(index_col)


### 特徴量の生成

In [7]:
agg_func_map={
    # "mean": pl.col("click").mean(),
    # "max": pl.col("click").max(),
    # "min": pl.col("click").min(),
    # "median": pl.col("click").median(),
    # "std": pl.col("click").std(),
    "count": pl.col("click").sum()
    # "n_unique": pl.col("click").n_unique(),
}

In [8]:
click_count_cols = ['A_click_count', 'B_click_count', 'C_click_count','D_click_count', 'E_click_count']

In [9]:
# ----------------------------
# ユーザー × サイトの特徴量
# ----------------------------
user_site_stats = aggregate_by_category_pivot(
    df=train_df,
    index_col=["user_id"],
    category_col="site",
    agg_func_map=agg_func_map,
    default_value=0.0
)

In [10]:
user_site_stats.columns

Index(['user_id', 'C_click_count', 'D_click_count', 'E_click_count',
       'B_click_count', 'A_click_count'],
      dtype='object')

In [11]:
user_site_stats = user_site_stats[['user_id'] + click_count_cols].sort_values('user_id').reset_index(drop=True)
user_site_stats = rename_columns_with_tag(user_site_stats, click_count_cols, 'user')

In [13]:
user_site_stats.head()

,user_id,A_click_count_user,B_click_count_user,C_click_count_user,D_click_count_user,E_click_count_user
0,user_0,3.0,3.0,1.0,1.0,1.0
1,user_1,5.0,3.0,3.0,1.0,3.0
2,user_2,1.0,1.0,0.0,2.0,2.0
3,user_3,3.0,1.0,2.0,1.0,2.0
4,user_4,1.0,1.0,2.0,2.0,5.0


In [14]:
# ----------------------------
# 年齢 × サイトの特徴量
# ----------------------------
age_site_stats = aggregate_by_category_pivot(
    df=train_df,
    index_col=["age_group"],
    category_col="site",
    agg_func_map=agg_func_map,
    default_value=0.0
)

In [15]:
age_site_stats.columns

Index(['age_group', 'A_click_count', 'B_click_count', 'C_click_count',
       'D_click_count', 'E_click_count'],
      dtype='object')

In [16]:
age_site_stats = age_site_stats[['age_group'] + click_count_cols].sort_values('age_group').reset_index(drop=True)
age_site_stats  = rename_columns_with_tag(age_site_stats , click_count_cols, 'age')

In [18]:
age_site_stats.head()

,age_group,A_click_count_age,B_click_count_age,C_click_count_age,D_click_count_age,E_click_count_age
0,20代,6.0,4.0,4.0,2.0,5.0
1,30代,10.0,6.0,4.0,5.0,8.0
2,40代,3.0,8.0,5.0,5.0,8.0
3,50代,2.0,6.0,5.0,1.0,3.0


In [19]:
# ----------------------------
# 性別 × サイトの特徴量
# ----------------------------
gender_site_stats = aggregate_by_category_pivot(
    df=train_df,
    index_col=["gender"],
    category_col="site",
    agg_func_map=agg_func_map,
    default_value=0.0
)

In [20]:
gender_site_stats.columns

Index(['gender', 'C_click_count', 'A_click_count', 'D_click_count',
       'E_click_count', 'B_click_count'],
      dtype='object')

In [21]:
gender_site_stats = gender_site_stats[['gender'] + click_count_cols].sort_values('gender').reset_index(drop=True)
gender_site_stats  = rename_columns_with_tag(gender_site_stats , click_count_cols, 'gender')

In [23]:
gender_site_stats.head()

,gender,A_click_count_gender,B_click_count_gender,C_click_count_gender,D_click_count_gender,E_click_count_gender
0,女性,12.0,11.0,8.0,4.0,12.0
1,男性,9.0,13.0,10.0,9.0,12.0


In [24]:
# ----------------------------
# (年齢 × 性別) × サイトの特徴量
# ----------------------------
age_gender_site_stats = aggregate_by_category_pivot(
    df=train_df,
    index_col=["age_group", "gender"],
    category_col="site",
    agg_func_map=agg_func_map,
    default_value=0.0
)

In [25]:
age_gender_site_stats.columns

Index(['age_group', 'gender', 'B_click_count', 'E_click_count',
       'A_click_count', 'D_click_count', 'C_click_count'],
      dtype='object')

In [26]:
age_gender_site_stats = age_gender_site_stats[["age_group", 'gender'] + click_count_cols].sort_values(["age_group", 'gender']).reset_index(drop=True)
age_gender_site_stats  = rename_columns_with_tag(age_gender_site_stats , click_count_cols, 'age_gender')

In [28]:
age_gender_site_stats

,age_group,gender,A_click_count_age_gender,B_click_count_age_gender,C_click_count_age_gender,D_click_count_age_gender,E_click_count_age_gender
0,20代,女性,4.0,3.0,2.0,1.0,3.0
1,20代,男性,2.0,1.0,2.0,1.0,2.0
2,30代,女性,5.0,2.0,1.0,1.0,3.0
3,30代,男性,5.0,4.0,3.0,4.0,5.0
4,40代,女性,2.0,5.0,0.0,1.0,4.0
5,40代,男性,1.0,3.0,5.0,4.0,4.0
6,50代,女性,1.0,1.0,5.0,1.0,2.0
7,50代,男性,1.0,5.0,0.0,0.0,1.0


### 特徴量を結合

In [29]:
train_df = pd.merge(train_df, user_site_stats, how="left" ,on=["user_id"])
train_df = pd.merge(train_df, age_site_stats, how="left" ,on=["age_group"])
train_df = pd.merge(train_df, gender_site_stats, how="left" ,on=["gender"])
train_df = pd.merge(train_df, age_gender_site_stats, how="left" ,on=["age_group", "gender"])


In [30]:
train_df.head()

,user_id,age_group,gender,site,click_date,click,A_click_count_user,B_click_count_user,C_click_count_user,D_click_count_user,...,A_click_count_gender,B_click_count_gender,C_click_count_gender,D_click_count_gender,E_click_count_gender,A_click_count_age_gender,B_click_count_age_gender,C_click_count_age_gender,D_click_count_age_gender,E_click_count_age_gender
0,user_3,20代,女性,C,2025-05-25,1,3.0,1.0,2.0,1.0,...,12.0,11.0,8.0,4.0,12.0,4.0,3.0,2.0,1.0,3.0
1,user_6,30代,女性,A,2025-05-25,1,1.0,3.0,6.0,0.0,...,12.0,11.0,8.0,4.0,12.0,5.0,2.0,1.0,1.0,3.0
2,user_4,30代,女性,D,2025-05-25,1,1.0,1.0,2.0,2.0,...,12.0,11.0,8.0,4.0,12.0,5.0,2.0,1.0,1.0,3.0
3,user_1,40代,男性,D,2025-05-25,1,5.0,3.0,3.0,1.0,...,9.0,13.0,10.0,9.0,12.0,1.0,3.0,5.0,4.0,4.0
4,user_9,30代,男性,E,2025-05-24,1,1.0,1.0,0.0,2.0,...,9.0,13.0,10.0,9.0,12.0,5.0,4.0,3.0,4.0,5.0


## 時系列特徴量のエンジニアリング

### 特徴量エンジニアリング関数の定義

In [31]:
@df_io_polars(return_type='pandas')
def compute_recency_frequency_features(
    df: pl.DataFrame,
    group_keys: Union[str, List[str]],
    date_col: str,
    site_col: str,
    reference_date: datetime,
    windows: List[int] = [30, 60, 90]
) -> pl.DataFrame:
    """
    For each (group_key(s) × site), compute:
      - recency: days since last click
      - frequency: counts in past N days
    """
    if isinstance(group_keys, str):
        group_keys = [group_keys]

    df = df.with_columns([
        pl.col(date_col).cast(pl.Date)
    ])

    # --- Recency ---
    recency_df = (
        df.group_by(group_keys + [site_col])
          .agg(pl.col(date_col).max().alias("last_click"))
          .with_columns([
              pl.lit(reference_date).cast(pl.Date).alias("ref_date"),
              (pl.lit(reference_date) - pl.col("last_click")).dt.total_days().alias("recency_days")
          ])
          .drop("last_click", "ref_date")
          .pivot(index=group_keys, columns=site_col, values="recency_days")
          .rename({col: f"recency_{col}_days" for col in df[site_col].unique()})
    )

    # --- Frequency ---
    freq_dfs = []
    for win in windows:
        min_date = reference_date - timedelta(days=win)
        freq = (
            df.filter(pl.col(date_col) >= pl.lit(min_date))
              .group_by(group_keys + [site_col])
              .agg(pl.len().alias("freq"))
              .pivot(index=group_keys, columns=site_col, values="freq")
              .rename({col: f"freq{win}_{col}_count" for col in df[site_col].unique()})
        )
        freq_dfs.append(freq)

    # --- Join all tables safely on group_keys ---
    result = recency_df
    for part in freq_dfs:
        dup_cols = set(result.columns) & set(part.columns)
        cols_to_remove = dup_cols - set(group_keys)
        part = part.drop(list(cols_to_remove)) if cols_to_remove else part
    
        # 外部結合から左結合へ変更して衝突回避
        result = result.join(part, on=group_keys, how="left")

    result = result.fill_null(0).sort(group_keys)
    return result


### 特徴量の生成

In [32]:
recency_cols = [f"recency_{s}_days" for s in sites]
recency_cols += [f"freq30_{s}_count" for s in sites]
recency_cols += [f"freq60_{s}_count" for s in sites]
recency_cols += [f"freq90_{s}_count" for s in sites]

In [33]:
reference_date

datetime.date(2025, 5, 25)

In [34]:
# ユーザー × サイトのRecency/Frequency
user_rec_freq = compute_recency_frequency_features(
    df=train_df[original_cols],
    group_keys=["user_id"],
    date_col="click_date",
    site_col="site",
    reference_date=reference_date,
    windows=[30, 60, 90]
)

user_rec_freq = user_rec_freq[["user_id"] + recency_cols]
user_rec_freq = rename_columns_with_tag(user_rec_freq, recency_cols, 'user')

user_rec_freq.head()

,user_id,recency_A_days_user,recency_B_days_user,recency_C_days_user,recency_D_days_user,recency_E_days_user,freq30_A_count_user,freq30_B_count_user,freq30_C_count_user,freq30_D_count_user,...,freq60_A_count_user,freq60_B_count_user,freq60_C_count_user,freq60_D_count_user,freq60_E_count_user,freq90_A_count_user,freq90_B_count_user,freq90_C_count_user,freq90_D_count_user,freq90_E_count_user
0,user_0,61,116,110,117,14,0,0,0,0,...,0,0,0,0,1,2,0,0,0,1
1,user_1,8,40,9,0,28,2,0,1,1,...,4,1,2,1,1,5,1,3,1,2
2,user_2,8,105,0,54,125,1,0,0,0,...,1,0,0,1,0,1,0,0,1,0
3,user_3,14,5,0,17,89,1,1,1,1,...,1,1,1,1,0,1,1,1,1,1
4,user_4,41,6,7,0,13,0,1,1,2,...,1,1,2,2,1,1,1,2,2,1


In [35]:
# 年齢 × サイトのRecency/Frequency
age_rec_freq = compute_recency_frequency_features(
    df=train_df[original_cols],
    group_keys=["age_group"],
    date_col="click_date",
    site_col="site",
    reference_date=reference_date,
    windows=[30, 60, 90]
)

age_rec_freq = age_rec_freq[["age_group"] + recency_cols]
age_rec_freq = rename_columns_with_tag(age_rec_freq, recency_cols, 'age')

age_rec_freq.head()

,age_group,recency_A_days_age,recency_B_days_age,recency_C_days_age,recency_D_days_age,recency_E_days_age,freq30_A_count_age,freq30_B_count_age,freq30_C_count_age,freq30_D_count_age,...,freq60_A_count_age,freq60_B_count_age,freq60_C_count_age,freq60_D_count_age,freq60_E_count_age,freq90_A_count_age,freq90_B_count_age,freq90_C_count_age,freq90_D_count_age,freq90_E_count_age
0,20代,8,5,0,19,107,3,1,1,1,...,5,2,2,1,0,6,2,3,1,0
1,30代,0,6,6,0,1,3,3,2,2,...,5,3,2,2,1,7,4,2,4,1
2,40代,57,33,55,0,28,0,0,0,3,...,1,3,2,3,3,2,4,3,3,4
3,50代,116,10,7,54,13,0,1,2,0,...,0,1,2,1,2,0,2,2,1,3


In [36]:
# 性別 × サイトのRecency/Frequency
gender_rec_freq = compute_recency_frequency_features(
    df=train_df[original_cols],
    group_keys=["gender"],
    date_col="click_date",
    site_col="site",
    reference_date=reference_date,
    windows=[30, 60, 90]
)

gender_rec_freq = gender_rec_freq[["gender"] + recency_cols]
gender_rec_freq = rename_columns_with_tag(gender_rec_freq, recency_cols, 'gender')

gender_rec_freq.head()

,gender,recency_A_days_gender,recency_B_days_gender,recency_C_days_gender,recency_D_days_gender,recency_E_days_gender,freq30_A_count_gender,freq30_B_count_gender,freq30_C_count_gender,freq30_D_count_gender,...,freq60_A_count_gender,freq60_B_count_gender,freq60_C_count_gender,freq60_D_count_gender,freq60_E_count_gender,freq90_A_count_gender,freq90_B_count_gender,freq90_C_count_gender,freq90_D_count_gender,freq90_E_count_gender
0,女性,0,5,0,0,13,5,2,4,2,...,6,6,5,3,3,9,7,5,3,4
1,男性,8,6,6,0,1,1,3,1,4,...,5,3,3,4,3,6,5,5,6,4


In [37]:
# 年齢 × 性別 × サイトのRecency/Frequency
age_gender_rec_freq = compute_recency_frequency_features(
    df=train_df[original_cols],
    group_keys=["age_group", "gender"],
    date_col="click_date",
    site_col="site",
    reference_date=reference_date,
    windows=[30, 60, 90]
)

age_gender_rec_freq = age_gender_rec_freq[["age_group", "gender"] + recency_cols]
age_gender_rec_freq = rename_columns_with_tag(age_gender_rec_freq, recency_cols, 'age_gender')

age_gender_rec_freq.head()

,age_group,gender,recency_A_days_age_gender,recency_B_days_age_gender,recency_C_days_age_gender,recency_D_days_age_gender,recency_E_days_age_gender,freq30_A_count_age_gender,freq30_B_count_age_gender,freq30_C_count_age_gender,...,freq60_A_count_age_gender,freq60_B_count_age_gender,freq60_C_count_age_gender,freq60_D_count_age_gender,freq60_E_count_age_gender,freq90_A_count_age_gender,freq90_B_count_age_gender,freq90_C_count_age_gender,freq90_D_count_age_gender,freq90_E_count_age_gender
0,20代,女性,8,5,0,97,125,3,1,1,...,3,2,2,0,0,4,2,2,0,0
1,20代,男性,39,124,67,19,107,0,0,0,...,2,0,0,1,0,2,0,1,1,0
2,30代,女性,0,12,9,0,110,2,1,1,...,3,1,1,1,0,4,2,1,1,0
3,30代,男性,8,6,6,24,1,1,2,1,...,2,2,1,1,1,3,2,1,3,1
4,40代,女性,61,33,0,17,28,0,0,0,...,0,3,0,1,2,1,3,0,1,2


### 特徴量を結合

In [38]:
train_df = pd.merge(train_df, user_rec_freq, how="left" ,on=["user_id"])
train_df = pd.merge(train_df, age_rec_freq, how="left" ,on=["age_group"])
train_df = pd.merge(train_df, gender_rec_freq, how="left" ,on=["gender"])
train_df = pd.merge(train_df, age_gender_rec_freq, how="left" ,on=["age_group", "gender"])


In [39]:
train_df.head()

,user_id,age_group,gender,site,click_date,click,A_click_count_user,B_click_count_user,C_click_count_user,D_click_count_user,...,freq60_A_count_age_gender,freq60_B_count_age_gender,freq60_C_count_age_gender,freq60_D_count_age_gender,freq60_E_count_age_gender,freq90_A_count_age_gender,freq90_B_count_age_gender,freq90_C_count_age_gender,freq90_D_count_age_gender,freq90_E_count_age_gender
0,user_3,20代,女性,C,2025-05-25,1,3.0,1.0,2.0,1.0,...,3,2,2,0,0,4,2,2,0,0
1,user_6,30代,女性,A,2025-05-25,1,1.0,3.0,6.0,0.0,...,3,1,1,1,0,4,2,1,1,0
2,user_4,30代,女性,D,2025-05-25,1,1.0,1.0,2.0,2.0,...,3,1,1,1,0,4,2,1,1,0
3,user_1,40代,男性,D,2025-05-25,1,5.0,3.0,3.0,1.0,...,1,0,2,2,1,1,1,3,2,2
4,user_9,30代,男性,E,2025-05-24,1,1.0,1.0,0.0,2.0,...,2,2,1,1,1,3,2,1,3,1


## 行列分解やWord2Vecによる潜在因子特徴量

### 特徴量エンジニアリング関数の定義

In [40]:
def compute_svd_features(
    df: pd.DataFrame,
    user_col: List[str],
    item_col: str = "site",
    n_components: int = 8
) -> pd.DataFrame:
    """
    Compute latent group-item features using Truncated SVD on interaction frequency.

    Args:
        df (pd.DataFrame): Input DataFrame containing interaction logs.
        user_col (List[str]): List of column names identifying user/group keys.
        item_col (str): Column representing item identity.
        n_components (int): Number of latent dimensions to extract.

    Returns:
        pd.DataFrame: A DataFrame with latent features and user/group columns.
    """
    df = df.copy()
    df["_group_key"] = df[user_col].astype(str).agg("::".join, axis=1)

    user_le = LabelEncoder()
    item_le = LabelEncoder()

    user_ids = user_le.fit_transform(df["_group_key"])
    item_ids = item_le.fit_transform(df[item_col])

    n_users = user_ids.max() + 1
    n_items = item_ids.max() + 1
    n_components = min(n_components, min(n_users, n_items))

    interaction_matrix = np.zeros((n_users, n_items))
    for u, i in zip(user_ids, item_ids):
        interaction_matrix[u, i] += 1

    svd = TruncatedSVD(n_components=n_components, random_state=42)
    user_vectors = svd.fit_transform(interaction_matrix)

    df_user_vecs = pd.DataFrame(user_vectors, columns=[f"svd_u2i_{i}" for i in range(n_components)])
    df_user_vecs["_group_key"] = user_le.inverse_transform(np.arange(n_users))

    # split group_key back to original columns
    df_user_vecs[user_col] = df_user_vecs["_group_key"].str.split("::", expand=True)
    df_user_vecs = df_user_vecs.drop(columns=["_group_key"])

    return df_user_vecs


In [41]:
def compute_w2v_features(
    df: pd.DataFrame,
    user_col: List[str],
    item_col: str = "site",
    date_col: str = "click_date",
    n_components: int = 8
) -> pd.DataFrame:
    """
    Compute latent group-item features using Word2Vec on click sequences.

    Args:
        df (pd.DataFrame): Click log data with group, item, and date columns.
        user_col (List[str]): List of columns identifying user/group keys.
        item_col (str): Item column used as tokens for Word2Vec.
        date_col (str): Datetime column for sorting interaction order.
        n_components (int): Dimensionality of Word2Vec embeddings.

    Returns:
        pd.DataFrame: A DataFrame with averaged Word2Vec vectors and group columns.
    """
    df = df.copy()
    df["_group_key"] = df[user_col].astype(str).agg("::".join, axis=1)

    sequences = (
        df.sort_values(date_col)
          .groupby("_group_key")[item_col]
          .apply(list)
          .tolist()
    )

    model = Word2Vec(
        sentences=sequences,
        vector_size=n_components,
        window=5,
        min_count=1,
        workers=1,
        seed=42
    )

    group_vecs = {}
    for group_id, group_df in df.groupby("_group_key"):
        items = group_df[item_col].tolist()
        vecs = [model.wv[item] for item in items if item in model.wv]
        group_vecs[group_id] = np.mean(vecs, axis=0) if vecs else np.zeros(n_components)

    df_vecs = pd.DataFrame.from_dict(group_vecs, orient="index", columns=[f"w2v_u2i_{i}" for i in range(n_components)])
    df_vecs["_group_key"] = df_vecs.index
    df_vecs = df_vecs.reset_index(drop=True)

    # split group_key back to original columns
    df_vecs[user_col] = df_vecs["_group_key"].str.split("::", expand=True)
    df_vecs = df_vecs.drop(columns=["_group_key"])

    return df_vecs


### 特徴量の生成

#### SVD

In [42]:
#　ユーザー × サイトのSVD
user_svd = compute_svd_features(train_df[original_cols], user_col=["user_id"], item_col="site", n_components=8)

svd_cols = [f"svd_u2i_{i}" for i in range(len(user_svd.columns) - 1)]
user_svd = user_svd[["user_id"] + svd_cols]
user_svd = rename_columns_with_tag(user_svd, svd_cols, 'user')

user_svd.head()

,user_id,svd_u2i_0_user,svd_u2i_1_user,svd_u2i_2_user,svd_u2i_3_user,svd_u2i_4_user
0,user_0,4.168417,0.636478,-1.759374,-0.285151,0.206123
1,user_1,6.956078,1.079701,-0.542320,-1.682579,-0.567487
2,user_2,2.521785,-1.751694,-0.211119,-0.353689,0.634431
3,user_3,4.033690,0.367150,0.278439,-1.585757,-0.048932
4,user_4,4.900081,-2.418691,2.215297,-0.307537,-0.370155


In [43]:
#年齢 × サイトのSVD
age_svd = compute_svd_features(train_df[original_cols], user_col=["age_group"], item_col="site", n_components=8)

svd_cols = [f"svd_u2i_{i}" for i in range(len(age_svd.columns) - 1)]
age_svd = age_svd[["age_group"] + svd_cols]
age_svd = rename_columns_with_tag(age_svd, svd_cols, 'age')

age_svd.head()

,age_group,svd_u2i_0_age,svd_u2i_1_age,svd_u2i_2_age,svd_u2i_3_age
0,20代,9.650104,1.370396,1.263799,0.632712
1,30代,15.092964,3.603574,-0.277280,-0.373896
2,40代,13.139823,-3.273007,-1.900933,0.137599
3,50代,7.748102,-3.175790,2.189841,-0.293048


In [44]:
# 性別 × サイトのSVD
gender_svd = compute_svd_features(train_df[original_cols], user_col=["gender"], item_col="site", n_components=8)

svd_cols = [f"svd_u2i_{i}" for i in range(len(gender_svd.columns) - 1)]
gender_svd = gender_svd[["gender"] + svd_cols]
gender_svd = rename_columns_with_tag(gender_svd, svd_cols, 'gender')

gender_svd.head()

,gender,svd_u2i_0_gender,svd_u2i_1_gender
0,女性,21.876889,3.225171
1,男性,23.795120,-2.965176


In [48]:
# (年齢 x 性別) × サイトのSVD
age_gender_svd = compute_svd_features(train_df[original_cols], user_col=["age_group", "gender"], item_col="site", n_components=8)

svd_cols = [f"svd_u2i_{i}" for i in range(len(age_gender_svd.columns) - 2)]
age_gender_svd = age_gender_svd[["age_group", "gender"] + svd_cols]
age_gender_svd = rename_columns_with_tag(age_gender_svd, svd_cols, 'age_gender')

age_gender_svd.head()

,age_group,gender,svd_u2i_0_age_gender,svd_u2i_1_age_gender,svd_u2i_2_age_gender,svd_u2i_3_age_gender,svd_u2i_4_age_gender
0,20代,女性,6.042823,-0.839129,1.022555,0.856974,-0.011646
1,20代,男性,3.585044,0.737184,0.690040,0.293567,0.204159
2,30代,女性,5.621638,-1.367758,2.531359,0.342682,0.034926
3,30代,男性,9.420255,-0.021422,0.946603,-1.069240,-0.467974
4,40代,女性,5.878039,-2.848031,-1.513819,-0.409504,0.937036


#### word2vec

In [49]:
#　ユーザー × サイトのw2v
user_w2v = compute_w2v_features(train_df[original_cols], user_col=["user_id"], item_col="site", date_col="click_date", n_components=8)

w2v_cols = [f"w2v_u2i_{i}" for i in range(len(user_w2v.columns) - 1)]
user_w2v = user_w2v[["user_id"] + w2v_cols]
user_w2v = rename_columns_with_tag(user_w2v, w2v_cols, 'user')

user_w2v.head()

,user_id,w2v_u2i_0_user,w2v_u2i_1_user,w2v_u2i_2_user,w2v_u2i_3_user,w2v_u2i_4_user,w2v_u2i_5_user,w2v_u2i_6_user,w2v_u2i_7_user
0,user_0,-0.038829,-0.051954,0.043200,0.031188,0.029182,0.028692,-0.027875,0.061369
1,user_1,-0.027228,-0.029733,0.039257,0.025953,0.016867,0.023060,-0.042908,0.048673
2,user_2,-0.080208,-0.005241,0.060700,-0.023383,0.034228,0.062809,-0.057565,0.057414
3,user_3,-0.025844,-0.017767,0.043271,0.011942,0.015486,0.022568,-0.052751,0.043328
4,user_4,-0.059037,0.022242,0.039333,-0.002123,0.016236,0.056214,-0.062005,0.032532


In [50]:
# 年齢 × サイトのw2v
age_w2v = compute_w2v_features(train_df[original_cols], user_col=["age_group"], item_col="site", date_col="click_date", n_components=8)

w2v_cols = [f"w2v_u2i_{i}" for i in range(len(age_w2v.columns) - 1)]
age_w2v = age_w2v[["age_group"] + w2v_cols]
age_w2v = rename_columns_with_tag(age_w2v, w2v_cols, 'age')

age_w2v.head()

,age_group,w2v_u2i_0_age,w2v_u2i_1_age,w2v_u2i_2_age,w2v_u2i_3_age,w2v_u2i_4_age,w2v_u2i_5_age,w2v_u2i_6_age,w2v_u2i_7_age
0,20代,-0.033468,-0.030014,0.038980,0.026468,0.022334,0.029091,-0.038042,0.047945
1,30代,-0.043479,-0.034397,0.047333,0.015158,0.027022,0.033643,-0.040594,0.056274
2,40代,-0.054854,-0.010660,0.035024,0.022053,0.029848,0.051725,-0.034505,0.039238
3,50代,-0.035177,0.006223,0.024585,0.031074,0.013907,0.040679,-0.044214,0.025425


In [51]:
#　性別 × サイトのw2v
gender_w2v = compute_w2v_features(train_df[original_cols], user_col=["gender"], item_col="site", date_col="click_date", n_components=8)

w2v_cols = [f"w2v_u2i_{i}" for i in range(len(gender_w2v.columns) - 1)]
gender_w2v = gender_w2v[["gender"] + w2v_cols]
gender_w2v = rename_columns_with_tag(gender_w2v, w2v_cols, 'gender')

gender_w2v.head()

,gender,w2v_u2i_0_gender,w2v_u2i_1_gender,w2v_u2i_2_gender,w2v_u2i_3_gender,w2v_u2i_4_gender,w2v_u2i_5_gender,w2v_u2i_6_gender,w2v_u2i_7_gender
0,女性,-0.041746,-0.022693,0.038961,0.025057,0.019650,0.036285,-0.043123,0.049660
1,男性,-0.047269,-0.016025,0.038447,0.020126,0.028605,0.043897,-0.037722,0.041452


In [52]:
#　(年齢 x 性別) × サイトのw2v
age_gender_w2v = compute_w2v_features(train_df[original_cols], user_col=["age_group", "gender"], item_col="site", date_col="click_date", n_components=8)

w2v_cols = [f"w2v_u2i_{i}" for i in range(len(age_gender_w2v.columns) - 2)]
age_gender_w2v = age_gender_w2v[["age_group", "gender"] + w2v_cols]
age_gender_w2v = rename_columns_with_tag(age_gender_w2v, w2v_cols, 'age_gender')

age_gender_w2v.head()

,age_group,gender,w2v_u2i_0_age_gender,w2v_u2i_1_age_gender,w2v_u2i_2_age_gender,w2v_u2i_3_age_gender,w2v_u2i_4_age_gender,w2v_u2i_5_age_gender,w2v_u2i_6_age_gender,w2v_u2i_7_age_gender
0,20代,女性,-0.037298,-0.029734,0.040596,0.024194,0.018419,0.030215,-0.042393,0.053216
1,20代,男性,-0.026139,-0.029477,0.034007,0.031348,0.026731,0.026387,-0.029604,0.038189
2,30代,女性,-0.037419,-0.048643,0.049851,0.019717,0.020405,0.023738,-0.042863,0.068080
3,30代,男性,-0.046247,-0.025609,0.044419,0.013274,0.029569,0.038756,-0.038445,0.048763
4,40代,女性,-0.076030,-0.019736,0.040900,0.022160,0.020152,0.060855,-0.043393,0.065431


### 特徴量を結合

#### SVD

In [53]:
train_df = pd.merge(train_df, user_svd, how="left" ,on=["user_id"])
train_df = pd.merge(train_df, age_svd, how="left" ,on=["age_group"])
train_df = pd.merge(train_df, gender_svd, how="left" ,on=["gender"])
train_df = pd.merge(train_df, age_gender_svd, how="left" ,on=["age_group", "gender"])


In [54]:
train_df.head()

,user_id,age_group,gender,site,click_date,click,A_click_count_user,B_click_count_user,C_click_count_user,D_click_count_user,...,svd_u2i_1_age,svd_u2i_2_age,svd_u2i_3_age,svd_u2i_0_gender,svd_u2i_1_gender,svd_u2i_0_age_gender,svd_u2i_1_age_gender,svd_u2i_2_age_gender,svd_u2i_3_age_gender,svd_u2i_4_age_gender
0,user_3,20代,女性,C,2025-05-25,1,3.0,1.0,2.0,1.0,...,1.370396,1.263799,0.632712,21.876889,3.225171,6.042823,-0.839129,1.022555,0.856974,-0.011646
1,user_6,30代,女性,A,2025-05-25,1,1.0,3.0,6.0,0.0,...,3.603574,-0.277280,-0.373896,21.876889,3.225171,5.621638,-1.367758,2.531359,0.342682,0.034926
2,user_4,30代,女性,D,2025-05-25,1,1.0,1.0,2.0,2.0,...,3.603574,-0.277280,-0.373896,21.876889,3.225171,5.621638,-1.367758,2.531359,0.342682,0.034926
3,user_1,40代,男性,D,2025-05-25,1,5.0,3.0,3.0,1.0,...,-3.273007,-1.900933,0.137599,23.795120,-2.965176,7.306195,3.121585,-1.784371,-0.825219,-0.101248
4,user_9,30代,男性,E,2025-05-24,1,1.0,1.0,0.0,2.0,...,3.603574,-0.277280,-0.373896,23.795120,-2.965176,9.420255,-0.021422,0.946603,-1.069240,-0.467974


#### word2vec

In [55]:
train_df = pd.merge(train_df, user_w2v, how="left" ,on=["user_id"])
train_df = pd.merge(train_df, age_w2v, how="left" ,on=["age_group"])
train_df = pd.merge(train_df, gender_w2v, how="left" ,on=["gender"])
train_df = pd.merge(train_df, age_gender_w2v, how="left" ,on=["age_group", "gender"])


In [56]:
train_df.head()

,user_id,age_group,gender,site,click_date,click,A_click_count_user,B_click_count_user,C_click_count_user,D_click_count_user,...,w2v_u2i_6_gender,w2v_u2i_7_gender,w2v_u2i_0_age_gender,w2v_u2i_1_age_gender,w2v_u2i_2_age_gender,w2v_u2i_3_age_gender,w2v_u2i_4_age_gender,w2v_u2i_5_age_gender,w2v_u2i_6_age_gender,w2v_u2i_7_age_gender
0,user_3,20代,女性,C,2025-05-25,1,3.0,1.0,2.0,1.0,...,-0.043123,0.049660,-0.037298,-0.029734,0.040596,0.024194,0.018419,0.030215,-0.042393,0.053216
1,user_6,30代,女性,A,2025-05-25,1,1.0,3.0,6.0,0.0,...,-0.043123,0.049660,-0.037419,-0.048643,0.049851,0.019717,0.020405,0.023738,-0.042863,0.068080
2,user_4,30代,女性,D,2025-05-25,1,1.0,1.0,2.0,2.0,...,-0.043123,0.049660,-0.037419,-0.048643,0.049851,0.019717,0.020405,0.023738,-0.042863,0.068080
3,user_1,40代,男性,D,2025-05-25,1,5.0,3.0,3.0,1.0,...,-0.037722,0.041452,-0.039052,-0.003598,0.029242,0.022714,0.035370,0.044589,-0.027196,0.019852
4,user_9,30代,男性,E,2025-05-24,1,1.0,1.0,0.0,2.0,...,-0.037722,0.041452,-0.046247,-0.025609,0.044419,0.013274,0.029569,0.038756,-0.038445,0.048763
